In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
RAW_FILE = Path("../data/raw/online_retail_II.xlsx")

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
df_1 = pd.read_excel(
    RAW_FILE,
    sheet_name="Year 2009-2010"
)

df_2 = pd.read_excel(
    RAW_FILE,
    sheet_name="Year 2010-2011"
)

print("Sheet 1:", df_1.shape)
print("Sheet 2:", df_2.shape)

Sheet 1: (525461, 8)
Sheet 2: (541910, 8)


In [4]:
df = pd.concat(
    [df_1, df_2],
    ignore_index=True
)

print("Combined:", df.shape)

Combined: (1067371, 8)


In [5]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [6]:
df.columns.tolist()

['invoice',
 'stockcode',
 'description',
 'quantity',
 'invoicedate',
 'price',
 'customer_id',
 'country']

In [7]:
df["customer_id"] = (
    df["customer_id"]
    .astype("Int64")
)

In [8]:
df["customer_id"].dtype

Int64Dtype()

In [9]:
df["invoice"] = df["invoice"].astype(str).str.strip()

In [10]:
df["stockcode"] = (
    df["stockcode"]
    .astype(str)
    .str.strip()
)

In [11]:
df["description"] = (
    df["description"]
    .astype("string")
    .str.strip()
)

In [12]:
df["country"] = (
    df["country"]
    .astype("string")
    .str.strip()
)

In [13]:
df["invoicedate"] = pd.to_datetime(
    df["invoicedate"],
    errors="coerce"
)

In [14]:
df["invoicedate"] = pd.to_datetime(
    df["invoicedate"],
    errors="coerce"
)

In [15]:
df.dtypes

invoice                object
stockcode              object
description    string[python]
quantity                int64
invoicedate    datetime64[ns]
price                 float64
customer_id             Int64
country        string[python]
dtype: object

In [16]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 34335


In [17]:
df = df.drop_duplicates().reset_index(drop=True)

In [18]:
print("Remaining duplicates:", df.duplicated().sum())
print("New shape:", df.shape)

Remaining duplicates: 0
New shape: (1033036, 8)


In [19]:
df["description"] = df["description"].fillna(
    "Unknown Product"
)

In [20]:
df["description"].isna().sum()

np.int64(0)

In [21]:
invalid_price = df[df["price"] <= 0]

invalid_price[
    [
        "invoice",
        "stockcode",
        "description",
        "quantity",
        "price",
        "customer_id"
    ]
].head(20)

,invoice,stockcode,description,quantity,price,customer_id
263,489464,21733,85123a mixed,-96,0.0,<NA>
283,489463,71477,short,-240,0.0,<NA>
284,489467,85123A,21733 mixed,-192,0.0,<NA>
462,489521,21646,Unknown Product,-50,0.0,<NA>
3077,489655,20683,Unknown Product,-44,0.0,<NA>
3124,489659,21350,Unknown Product,230,0.0,<NA>
3125,489660,35956,lost,-1043,0.0,<NA>
3131,489663,35605A,damages,-117,0.0,<NA>
3687,489781,84292,Unknown Product,17,0.0,<NA>
4233,489806,18010,Unknown Product,-770,0.0,<NA>


In [22]:
print("Invalid price rows:", len(invalid_price))

print(
    "Invalid price revenue:",
    (invalid_price["quantity"] * invalid_price["price"]).sum()
)

Invalid price rows: 6019
Invalid price revenue: -158676.14


In [23]:
df["invalid_price_flag"] = df["price"] <= 0

In [24]:
df_sales = df[df["price"] > 0].copy()

In [25]:
print(df_sales.shape)
print((df_sales["price"] <= 0).sum())

(1027017, 9)
0


In [26]:
df["is_cancellation"] = (
    df["invoice"]
    .str.upper()
    .str.startswith("C")
)

In [27]:
df["is_cancellation"].value_counts()

is_cancellation
False    1013932
True       19104
Name: count, dtype: int64

In [28]:
df[
    df["is_cancellation"]
][
    [
        "invoice",
        "stockcode",
        "quantity",
        "price",
        "customer_id"
    ]
].head(20)

,invoice,stockcode,quantity,price,customer_id
178,C489449,22087,-12,2.95,16321
179,C489449,85206A,-6,1.65,16321
180,C489449,21895,-4,4.25,16321
181,C489449,21896,-6,2.10,16321
182,C489449,22083,-12,2.95,16321
183,C489449,21871,-12,1.25,16321
184,C489449,84946,-12,1.25,16321
185,C489449,84970S,-24,0.85,16321
186,C489449,22090,-12,2.95,16321
196,C489459,90200A,-3,4.25,17592


In [29]:
pd.crosstab(
    df["is_cancellation"],
    df["quantity"] < 0
)

quantity,False,True
is_cancellation,,
False,1010539,3393
True,1,19103


In [30]:
df["is_return"] = df["quantity"] < 0

In [31]:
df["is_return"].value_counts()

is_return
False    1010540
True       22496
Name: count, dtype: int64

In [32]:
df["revenue"] = (
    df["quantity"] * df["price"]
)

In [33]:
df["revenue"].describe()

count    1.033036e+06
mean     1.825254e+01
std      2.956873e+02
min     -1.684696e+05
25%      3.750000e+00
50%      9.920000e+00
75%      1.770000e+01
max      1.684696e+05
Name: revenue, dtype: float64

In [34]:
df_sales = df[
    (df["price"] > 0) &
    (df["quantity"] > 0) &
    (~df["is_cancellation"])
].copy()

In [35]:
print(df_sales.shape)

(1007913, 12)


In [36]:
print(
    "Customers:",
    df_sales["customer_id"].nunique()
)

Customers: 5878


In [37]:
df_customer = df_sales[
    df_sales["customer_id"].notna()
].copy()

In [38]:
print("Customer dataset:", df_customer.shape)
print(
    "Unique customers:",
    df_customer["customer_id"].nunique()
)

Customer dataset: (779425, 12)
Unique customers: 5878


In [39]:
print("Missing customer IDs:",
      df_customer["customer_id"].isna().sum())

print("Missing dates:",
      df_customer["invoicedate"].isna().sum())

print("Invalid prices:",
      (df_customer["price"] <= 0).sum())

print("Invalid quantities:",
      (df_customer["quantity"] <= 0).sum())

print("Duplicate rows:",
      df_customer.duplicated().sum())

Missing customer IDs: 0
Missing dates: 0
Invalid prices: 0
Invalid quantities: 0
Duplicate rows: 0


In [40]:
df_sales.to_csv(
    PROCESSED_DIR / "retail_sales_cleaned.csv",
    index=False
)

In [41]:
df_customer.to_csv(
    PROCESSED_DIR / "customer_transactions.csv",
    index=False
)

In [42]:
df_sales.shape

(1007913, 12)

In [43]:
df_customer.shape

(779425, 12)

In [44]:
df_customer["customer_id"].nunique()

5878

In [45]:
df["is_cancellation"].value_counts()

is_cancellation
False    1013932
True       19104
Name: count, dtype: int64

In [46]:
pd.crosstab(df["is_cancellation"], df["quantity"] < 0)

quantity,False,True
is_cancellation,,
False,1010539,3393
True,1,19103


In [47]:
df["is_return"].value_counts()

is_return
False    1010540
True       22496
Name: count, dtype: int64

In [48]:
df["revenue"].describe()

count    1.033036e+06
mean     1.825254e+01
std      2.956873e+02
min     -1.684696e+05
25%      3.750000e+00
50%      9.920000e+00
75%      1.770000e+01
max      1.684696e+05
Name: revenue, dtype: float64